## Custom Hyperparameter Tuning Loop

When tuning hyperparameters, you need a way to compare model configurations
**without** peeking at the test set. This notebook walks through how to do that
by hand using a **three-way split** and a simple loop.

### Why a three-way split?

With only a **train/test** split you face a dilemma:

| If you tune on … | Problem |
|---|---|
| **Training set** | Every model looks great on data it already memorized. |
| **Test set** | You indirectly leak test information into your model choice, so the reported accuracy is optimistic. |

The fix is to carve out a **validation set** — a slice of data the model never
trains on but that you *are* allowed to peek at repeatedly while tuning.

```
┌──────────────┬──────────────┬──────────────┐
│   Train 60%  │   Val 20%    │  Test 20%    │
│  (fit model) │ (pick best   │ (final grade,│
│              │  settings)   │  report once)│
└──────────────┴──────────────┴──────────────┘
```

Only **after** all tuning is done do you evaluate the winner on the test set.
That single number is the honest estimate of real-world performance.

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
print(f"Dataset: {len(X)} samples, {X.shape[1]} features, {len(np.unique(y))} classes")

Best depth: 4  |  Val accuracy: 0.933
Test accuracy: 0.933


### Step 1 — Three-way split (60 / 20 / 20)

We shuffle the row indices, then slice them into three non-overlapping groups.
Shuffling is critical because Iris ships with all class-0 samples first, then
class-1, then class-2 — a sequential split would give each partition only one class.

In [ ]:
np.random.seed(42)
idx = np.random.permutation(len(X))

train_end = int(0.6 * len(X))  # first 60%
val_end   = int(0.8 * len(X))  # next  20%
                                 # rest  20% → test

X_train, y_train = X[idx[:train_end]],        y[idx[:train_end]]
X_val,   y_val   = X[idx[train_end:val_end]],  y[idx[train_end:val_end]]
X_test,  y_test  = X[idx[val_end:]],            y[idx[val_end:]]

print(f"Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}")

### Step 2 — Tune `max_depth` on the validation set

We train a decision tree for every candidate depth on the **training** set,
then score it on the **validation** set. The depth with the highest validation
accuracy wins. The test set is untouched during this whole loop.

In [ ]:
results = {}

for depth in range(1, 16):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    results[depth] = accuracy_score(y_val, clf.predict(X_val))

best_depth = max(results, key=results.get)

print("depth  val_acc")
print("─────  ───────")
for d, acc in results.items():
    marker = " ◀ best" if d == best_depth else ""
    print(f"  {d:2d}    {acc:.3f}{marker}")

### Step 3 — Final evaluation on the test set

Now — and only now — we train the chosen model and measure accuracy on the
**test** set. Because this set was never used to make any decision, the
resulting number is an unbiased estimate of how the model will perform on
genuinely new data.

In [ ]:
final_clf = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
final_clf.fit(X_train, y_train)

val_acc  = accuracy_score(y_val,  final_clf.predict(X_val))
test_acc = accuracy_score(y_test, final_clf.predict(X_test))

print(f"Best max_depth : {best_depth}")
print(f"Val  accuracy  : {val_acc:.3f}")
print(f"Test accuracy  : {test_acc:.3f}")

### Key Takeaways

* **Train** set teaches the model.
* **Validation** set picks the best hyperparameters — you may evaluate it many times.
* **Test** set gives the final, honest score — evaluate it **once**.
* If validation and test accuracy are close, you can trust the result.
  A large gap (val >> test) suggests overfitting to the validation set,
  and you may need more data or cross-validation.